# Customer Analytics — Portfolio Project

**Objective:** analyze customer demographics, purchasing behavior, geography, customer tiers, and revenue concentration to produce actionable business insights.

## 1. Business Questions
- How large is the customer base and how much revenue does it generate?
- Which customer tiers and demographics have the highest value?
- Which locations contribute the most revenue?
- Does order frequency relate to spending?
- How concentrated is revenue among high-value customers?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')

In [ ]:
df = pd.read_csv('../data/customers.csv')
df['Registration_Date'] = pd.to_datetime(df['Registration_Date'], errors='coerce')
df['Average_Order_Value'] = np.where(df['Total_Orders'] > 0, df['Total_Spent'] / df['Total_Orders'], 0)
df.head()

## 2. Data Quality

In [ ]:
print('Shape:', df.shape)
print('Missing values:', df.isna().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate Customer IDs:', df['Customer_ID'].duplicated().sum())
df.info()

## 3. Executive KPIs

In [ ]:
total_customers = df['Customer_ID'].nunique()
total_orders = df['Total_Orders'].sum()
total_revenue = df['Total_Spent'].sum()
avg_customer_spend = df['Total_Spent'].mean()
avg_orders = df['Total_Orders'].mean()
avg_order_value = total_revenue / total_orders
zero_order_customers = (df['Total_Orders'] == 0).sum()

kpis = pd.Series({
    'Total Customers': total_customers,
    'Total Orders': total_orders,
    'Total Revenue': total_revenue,
    'Average Customer Spend': avg_customer_spend,
    'Average Orders per Customer': avg_orders,
    'Average Order Value': avg_order_value,
    'Zero-Order Customers': zero_order_customers
})
kpis

## 4. Customer Tier Analysis

In [ ]:
tier = df.groupby('Customer_Tier').agg(
    Customers=('Customer_ID','nunique'),
    Orders=('Total_Orders','sum'),
    Revenue=('Total_Spent','sum'),
    Avg_Spend=('Total_Spent','mean')
)
tier['Customer_Share_%'] = tier['Customers'] / total_customers * 100
tier['Revenue_Share_%'] = tier['Revenue'] / total_revenue * 100
tier.sort_values('Revenue', ascending=False)

In [ ]:
sns.barplot(data=tier.reset_index(), x='Customer_Tier', y='Revenue')
plt.title('Revenue by Customer Tier')
plt.show()

## 5. Demographic Analysis

In [ ]:
bins = [17,25,35,45,55,65,100]
labels = ['18-25','26-35','36-45','46-55','56-65','65+']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)
age = df.groupby('Age_Group', observed=False).agg(
    Customers=('Customer_ID','nunique'),
    Revenue=('Total_Spent','sum'),
    Avg_Spend=('Total_Spent','mean')
)
age

In [ ]:
sns.barplot(data=age.reset_index(), x='Age_Group', y='Avg_Spend')
plt.title('Average Customer Spend by Age Group')
plt.show()

## 6. Geographic Analysis

In [ ]:
state = df.groupby('State').agg(
    Customers=('Customer_ID','nunique'),
    Orders=('Total_Orders','sum'),
    Revenue=('Total_Spent','sum'),
    Avg_Spend=('Total_Spent','mean')
).sort_values('Revenue', ascending=False)
state

In [ ]:
sns.barplot(data=state.sort_values('Revenue').reset_index(), x='Revenue', y='State')
plt.title('Revenue by State')
plt.show()

## 7. Relationships and Correlations

In [ ]:
correlation = df[['Age','Total_Orders','Total_Spent']].corr()
correlation

In [ ]:
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Customer Metrics Correlation')
plt.show()

In [ ]:
sns.scatterplot(data=df, x='Total_Orders', y='Total_Spent', hue='Customer_Tier', alpha=.5)
plt.title('Orders vs Customer Spending')
plt.show()

## 8. Revenue Concentration

In [ ]:
df_sorted = df.sort_values('Total_Spent', ascending=False).reset_index(drop=True)
for percentage in [0.01, 0.05, 0.10, 0.20]:
    count = int(len(df) * percentage)
    revenue_share = df_sorted.head(count)['Total_Spent'].sum() / total_revenue * 100
    print(f'Top {percentage:.0%}: {revenue_share:.2f}% of revenue')

## 9. Business Interpretation

### Key takeaways
- High-value customers account for a disproportionately large share of revenue.
- Order frequency and spending have a positive but imperfect relationship, so average order value should also be monitored.
- Demographic and geographic analysis can support targeted retention and acquisition.
- Customer tier should not be treated as a causal driver if the tier is defined using spending itself.

### Recommended actions
1. Prioritize retention for high-value customers.
2. Develop targeted campaigns to increase repeat purchases and average order value.
3. Use state/city performance to prioritize acquisition and retention.
4. Track customer KPIs regularly in a dashboard.